In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(r'C:\Users\konta\Documents\DIV_Academy\Module2(From_29_nov)\data\housing.csv')
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [3]:
outliers1 = df['median_house_value'] == df['median_house_value'].max()
outliers2 = df['housing_median_age'] == df['housing_median_age'].max()
outliers = outliers1 | outliers2

In [4]:
outliers

0        False
1        False
2         True
3         True
4         True
         ...  
20635    False
20636    False
20637    False
20638    False
20639    False
Length: 20640, dtype: bool

##### Q1 = df[col].quantile(0.25)
##### Q3 = df[col].quantile(0.75)
##### IQR = Q3 - Q1

##### outliers = (df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)

In [5]:
df = df.loc[~outliers, :].copy()
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY
15,-122.26,37.85,50.0,1120.0,283.0,697.0,264.0,2.1250,140000.0,NEAR BAY
18,-122.26,37.84,50.0,2239.0,455.0,990.0,419.0,1.9911,158700.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [6]:
X = df.drop('median_house_value', axis='columns')
y = df['median_house_value']

In [7]:
X['income_cat'] = pd.cut(
    X['median_income'],
    bins=[0, 1.5, 3.0, 4.5, 6, np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [8]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=X['income_cat'],
    test_size=0.2
)

In [9]:
X_train.drop('income_cat', axis='columns', inplace=True)
X_test.drop('income_cat', axis='columns', inplace=True)

In [10]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

n_clusters = 10
kmeans_ = KMeans(n_clusters=n_clusters)
kmeans_.fit(X_train[['latitude', 'longitude']])
print(kmeans_.cluster_centers_)

[[  32.97048257 -116.90010724]
 [  37.95204927 -120.89361702]
 [  34.05124619 -118.28567472]
 [  38.65388132 -121.78844427]
 [  36.60379826 -119.61088418]
 [  40.54941799 -123.70148148]
 [  37.56412556 -122.10163677]
 [  34.96338658 -119.71011182]
 [  33.92279811 -117.63773569]
 [  40.17976945 -121.80115274]]


In [11]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(self.n_clusters)
        self.kmeans_.fit(X)
        return self  

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=1).round(3)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [12]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

cat_cols = ['ocean_proximity']
log_cols = [
    'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'
    ]

cat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
log_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out='one-to-one')),
    ('scl', StandardScaler())
])

def ratio_func(arr_2d):
    return arr_2d[:, [0]] / arr_2d[:, [1]]
def ratio_feature_names(*args, **kwargs):
    return ['ratio']

rat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('newcol', FunctionTransformer(ratio_func, feature_names_out=ratio_feature_names)),
    ('scl', StandardScaler())
])
cluster_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('cluster', ClusterSimilarity(n_clusters=10))
])
num_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler())
])

preprocessing = ColumnTransformer(
    transformers=[
        ('CAT', cat_pipeline, cat_cols),
        ('LOG', log_pipeline, log_cols),
        ('RAT_b/r', rat_pipeline, ['total_bedrooms', 'total_rooms']),
        ('RAT_r/h', rat_pipeline, ['total_rooms', 'households']),
        ('RAT_p/h', rat_pipeline, ['population', 'households']),
        ("GEO", cluster_pipeline, ["latitude", "longitude"]),
    ], remainder=num_pipeline
)

X_train_arr = preprocessing.fit_transform(X_train)
X_train_prepared = pd.DataFrame(X_train_arr, columns=preprocessing.get_feature_names_out())

X_test_arr = preprocessing.fit_transform(X_test)
X_test_prepared = pd.DataFrame(X_test_arr, columns=preprocessing.get_feature_names_out())

X_train_prepared

,CAT__ocean_proximity_<1H OCEAN,CAT__ocean_proximity_INLAND,CAT__ocean_proximity_ISLAND,CAT__ocean_proximity_NEAR BAY,CAT__ocean_proximity_NEAR OCEAN,LOG__total_rooms,LOG__total_bedrooms,LOG__population,LOG__households,LOG__median_income,RAT_b/r__ratio,RAT_r/h__ratio,RAT_p/h__ratio,GEO__Cluster 0 similarity,GEO__Cluster 1 similarity,GEO__Cluster 2 similarity,GEO__Cluster 3 similarity,GEO__Cluster 4 similarity,GEO__Cluster 5 similarity,GEO__Cluster 6 similarity,GEO__Cluster 7 similarity,GEO__Cluster 8 similarity,GEO__Cluster 9 similarity,remainder__housing_median_age
0,1.0,0.0,0.0,0.0,0.0,0.261498,0.511116,0.766891,0.525280,-0.415982,0.546063,-0.464054,0.060087,0.978,0.000,0.001,0.000,0.044,0.000,0.000,0.000,0.360,0.018,1.310058
1,1.0,0.0,0.0,0.0,0.0,1.632886,2.016651,1.603930,2.030238,-0.207498,0.799161,-0.580053,-0.140572,0.000,0.825,0.001,0.001,0.000,0.000,0.329,0.064,0.000,0.000,-0.696878
2,0.0,0.0,0.0,0.0,1.0,-0.757101,-0.990575,-1.188927,-0.924007,0.253863,-0.585836,0.143989,-0.108515,0.000,0.887,0.000,0.009,0.000,0.000,0.102,0.069,0.000,0.000,1.833606
3,1.0,0.0,0.0,0.0,0.0,-1.410396,-1.287247,-0.298078,-1.163371,-0.356417,0.351565,-0.503859,0.356315,0.898,0.000,0.005,0.000,0.017,0.000,0.000,0.000,0.292,0.036,1.397316
4,0.0,1.0,0.0,0.0,0.0,0.862319,0.992780,1.204605,1.008442,-0.301613,0.148556,-0.278396,0.044892,0.467,0.000,0.000,0.000,0.269,0.000,0.000,0.000,0.999,0.000,0.001186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14852,1.0,0.0,0.0,0.0,0.0,-0.415104,-0.681924,-0.639949,-0.663634,0.959652,-0.681317,0.308298,-0.022448,0.000,0.239,0.000,0.096,0.000,0.003,0.010,0.050,0.000,0.000,0.524735
14853,0.0,0.0,0.0,1.0,0.0,0.231455,0.115741,0.057633,-0.038873,0.190491,-0.402251,0.384137,0.004404,0.000,0.888,0.000,0.028,0.000,0.000,0.290,0.278,0.000,0.000,0.786509
14854,0.0,0.0,0.0,1.0,0.0,-0.004581,-0.175028,0.336916,-0.053011,0.685029,-0.505890,-0.020154,0.118065,0.000,0.981,0.000,0.005,0.000,0.000,0.329,0.132,0.000,0.000,-0.260588
14855,1.0,0.0,0.0,0.0,0.0,-0.544942,-0.914471,-0.852674,-0.821324,1.639014,-0.878784,0.354031,-0.040648,0.000,0.812,0.001,0.001,0.000,0.000,0.364,0.070,0.000,0.000,0.786509


In [13]:
from sklearn.linear_model import LinearRegression

lin_reg = Pipeline([
    ('cleaning', preprocessing),
    ('linreg', LinearRegression())
])

lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

In [14]:
from sklearn.metrics import root_mean_squared_error

lin_rmse = root_mean_squared_error(y_test, y_pred)
lin_rmse

58077.21871756289

In [15]:
from sklearn.svm import SVR

svr = Pipeline([
    ("cleaning", preprocessing), 
    ("svr", SVR(kernel='rbf'))
    ])
svr.fit(X_train, y_train)
y_pred = svr.predict(X_test)
svr_rmse = root_mean_squared_error(y_test, y_pred)
svr_rmse

95582.8264747497

In [16]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("treereg", DecisionTreeRegressor())
    ])
tree_reg.fit(X_train, y_train)
y_pred = tree_reg.predict(X_test)
tree_rmse = root_mean_squared_error(y_test, y_pred)
tree_rmse

62263.97803681767

In [17]:
from sklearn.neighbors import KNeighborsRegressor

knn_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("knn_reg", KNeighborsRegressor())
    ])
knn_reg.fit(X_train, y_train)
y_pred = knn_reg.predict(X_test)
knn_rmse = root_mean_squared_error(y_test, y_pred)
knn_rmse

52144.57494470237

In [18]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", RandomForestRegressor())
    ])
forest_reg.fit(X_train, y_train)
y_pred = forest_reg.predict(X_test)
tree_rmse = root_mean_squared_error(y_test, y_pred)
tree_rmse

41892.25956095487

In [19]:
from sklearn.ensemble import VotingRegressor

voting_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", VotingRegressor([
		('lr', KNeighborsRegressor()),
		('rf', RandomForestRegressor(random_state=123))
    	])
	)
])
voting_reg.fit(X_train, y_train)
y_pred = voting_reg.predict(X_test)
voting_rmse = root_mean_squared_error(y_test, y_pred)
voting_rmse

44616.48107221614

In [20]:
from sklearn.ensemble import GradientBoostingRegressor

gb_reg = forest_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", GradientBoostingRegressor(
        n_estimators=10,
        max_depth=10,)
    )
])
gb_reg.fit(X_train, y_train)
y_pred = gb_reg.predict(X_test)
gb_rmse = root_mean_squared_error(y_test, y_pred)
gb_rmse

53354.28317577066

In [21]:
from sklearn.ensemble import BaggingRegressor

bag_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("knn_reg", BaggingRegressor(
		KNeighborsRegressor(3), 
		n_estimators=500,
		max_samples=len(X_train) // 4, 
		bootstrap=True,
		n_jobs=-1, 
		random_state=123)
    )]
)
bag_reg.fit(X_train, y_train)
y_pred = bag_reg.predict(X_test)
bag_rmse = root_mean_squared_error(y_test, y_pred)
bag_rmse

50932.96233845995

In [22]:
from sklearn.ensemble import AdaBoostRegressor

adab_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("adab_reg", AdaBoostRegressor(
		KNeighborsRegressor(3), 
		n_estimators=100,
		random_state=123)
    )]
)
adab_reg.fit(X_train, y_train)
y_pred = adab_reg.predict(X_test)
adab_rmse = root_mean_squared_error(y_test, y_pred)
adab_rmse

63361.49998441727

In [23]:
from sklearn.model_selection import cross_val_score

forest_rmses = -cross_val_score(
    forest_reg, 
    X.drop('income_cat', axis='columns', errors='ignore'), 
    y,
	scoring="neg_root_mean_squared_error", 
    cv=10)
forest_rmses, forest_rmses.mean()

(array([55756.11042246, 54843.41544468, 54915.5837381 , 54538.09098863,
        52768.56792933, 52866.66546898, 52530.90217423, 53411.10134905,
        54550.46834218, 54942.44276978]),
 np.float64(54112.33486274112))

In [24]:
voting_rmses = -cross_val_score(
    voting_reg, 
    X.drop('income_cat', axis='columns', errors='ignore'), 
    y,
	scoring="neg_root_mean_squared_error", 
    cv=10
    )
voting_rmses, voting_rmses.mean()

(array([46320.80777294, 44831.72120849, 46216.59623859, 45159.90202323,
        42884.16861594, 43675.12539646, 43621.10862821, 43139.89390369,
        45925.76211196, 44755.10978607]),
 np.float64(44653.019568558426))

### Fine-Tune Your Model

#### Grid Search

In [28]:
from sklearn.model_selection import GridSearchCV

full_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('random_forest', RandomForestRegressor(random_state=123))
])

param_grid = [
    {
        'preprocessing__GEO__cluster__n_clusters': [5, 8, 10],
        'random_forest__max_features': [4, 6, 8]
    },
    {
        'preprocessing__GEO__cluster__n_clusters': [10, 15],
        'random_forest__max_features': [6, 8, 10]
    }
]

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error'
)

grid_search.fit(X_train, y_train)

,estimator,Pipeline(step..._state=123))])
,param_grid,"[{'preprocessing__GEO__cluster__n_clusters': [5, 8, ...], 'random_forest__max_features': [4, 6, ...]}, {'preprocessing__GEO__cluster__n_clusters': [10, 15], 'random_forest__max_features': [6, 8, ...]}]"
,scoring,'neg_root_mean_squared_error'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('CAT', ...), ('LOG', ...), ...]"


In [29]:
grid_search.best_params_

{'preprocessing__GEO__cluster__n_clusters': 15,
 'random_forest__max_features': 6}

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)

cv_res = cv_res[[
    "param_preprocessing__GEO__cluster__n_clusters",
    "param_random_forest__max_features",
    "split0_test_score",
    "split1_test_score",
    "split2_test_score",
    "mean_test_score"
]]